# MOVIE-RAJA GUI — build on Google Colab

Builds the MOVIE-RAJA + MovieBox-TUI integration **on this Colab VM** and gives you two artifacts to download:

| Artifact | What it is |
| --- | --- |
| `dist/pkg.tar.zst` | **Linux desktop package** — Tauri app binary + MovieBox-TUI adapter + launcher + fallback web UI |
| `dist/moviera-android-debug.apk` | **Android APK** (installable, debug-signed; release signing optional) |
| `dist/linux-AppImage/`, `dist/linux-deb/` | AppImage / .deb bundles (when produced) |

Everything is best-effort per step with an OK/FAIL summary at the end.
Long-running cells may hit Colab's 10-minute cell timeout — the build itself
runs in the background (`nohup`) and survives; just re-run the watch cell.


In [ ]:
# 1) Get the repository (idempotent)
import os, subprocess

def run(cmd):
    print("$", cmd)
    return subprocess.call(cmd, shell=True)

if not os.path.exists("build_colab.py"):
    run("git clone --depth 1 https://github.com/samasadul124-ui/CassielDrive .")
run("git pull --ff-only || true")
print(os.listdir("."))


In [ ]:
# 2) GitHub PAT — plain text, per project requirement
#    Put your GitHub Personal Access Token as plain text below (in this variable),
#    it is written to .github_pat.txt which build_colab.py reads directly.
GITHUB_PAT = "PASTE_YOUR_GITHUB_PAT_HERE"

with open(".github_pat.txt", "w") as f:
    f.write(GITHUB_PAT.strip())
print("wrote .github_pat.txt (%d chars)" % len(GITHUB_PAT.strip()))


In [ ]:
# 3) Sanity: disk space (Android toolchain needs ~8 GB free)
import shutil
free = shutil.disk_usage("/").free / 1e9
print(f"free disk: {free:.1f} GB")
if free < 8:
    print("WARNING: less than 8 GB free — the Android step may not complete. "
          "Linux pkg.tar.zst should still build. You can skip android with --steps linux-only below.")


In [ ]:
# 4) START the build in the background (survives cell timeouts)
#    Change '--steps all' to e.g. '--steps deps,rust,node,frontend-deps,frontend-build,adapter,linux,collect'
#    to skip the Android step.
!nohup python build_colab.py --steps all > build.log 2>&1 &
!sleep 2 && tail -n 5 build.log


In [ ]:
# 5) WATCH the build (prints progress; re-run this cell if it times out —
#    the build keeps running in the background). Stops on its own at the summary.
import subprocess, time

deadline = time.time() + 9 * 60  # stay just under Colab's 10-minute cell limit
while time.time() < deadline:
    with open("build.log") as f:
        log = f.read()
    if "BUILD SUMMARY" in log:
        break
    tail = log.strip().splitlines()[-3:]
    print(" | ".join(t.tail if hasattr(t, "tail") else t for t in tail))
    time.sleep(30)

print("---- last 80 lines ----")
subprocess.run("tail -n 80 build.log", shell=True)


In [ ]:
# 6) DOWNLOAD the artifacts
from google.colab import files
import glob, os

artifacts = []
for p in ["dist/pkg.tar.zst"]:
    if os.path.exists(p):
        artifacts.append(p)
for pattern in ["dist/*.apk", "dist/linux-AppImage/**/*.AppImage", "dist/linux-deb/**/*.deb"]:
    artifacts += sorted(glob.glob(pattern, recursive=True))

if not artifacts:
    print("No artifacts yet — check the build log above (re-run step 5 if it timed out).")
else:
    for p in artifacts:
        print(p)
        files.download(p)


## Optional: run steps individually (for debugging)

Short steps run directly; long steps use the same nohup+tail pattern.

```python
!python build_colab.py --steps deps
!python build_colab.py --steps rust
!python build_colab.py --steps node
!python build_colab.py --steps update-repos
!python build_colab.py --steps frontend-deps
!python build_colab.py --steps frontend-build
```

Long steps (each in its own cell, then re-run the tail cell until done):

```python
!nohup python build_colab.py --steps adapter > build-adapter.log 2>&1 &
!tail -f build-adapter.log | grep -m1 'FAIL\|OK' ; tail -n 30 build-adapter.log
```

```python
!nohup python build_colab.py --steps linux > build-linux.log 2>&1 &
!tail -f build-linux.log | grep -m1 'FAIL\|OK' ; tail -n 30 build-linux.log
```

```python
!nohup python build_colab.py --steps android > build-android.log 2>&1 &
!tail -f build-android.log | grep -m1 'FAIL\|OK' ; tail -n 30 build-android.log
```

```python
!python build_colab.py --steps collect   # packages pkg.tar.zst + copies APK into dist/
```


## Troubleshooting

| Problem | Fix |
| --- | --- |
| `apt-get` permission errors | Colab runs as root — the script handles this; if not, prefix with `sudo` |
| `apt-get update` 404s | re-run the deps cell once (mirror hiccup) |
| Disk full during Android step | `!df -h` ; free space with `!rm -rf /tmp/*.zip ~/.gradle/caches` and re-run just `android` |
| `tauri build` fails on webkit headers | `!apt-get install -y libwebkit2gtk-4.1-dev libgtk-3-dev libayatana-appindicator3-dev libjavascriptcoregtk-4.1-dev` then re-run `linux` |
| Gradle OOM | the helper sets `GRADLE_OPTS=-Xmx3072m`; on Pro with more RAM you can raise it |
| APK is debug-signed | expected by default. For a **release** APK set the keystore env vars in a cell, then re-run the android step: `!export TAURI_ANDROID_SIGNING_KEYSTORE=/path/key.jks TAURI_ANDROID_SIGNING_KEYSTORE_PASSWORD=... TAURI_ANDROID_SIGNING_KEY_ALIAS=... TAURI_ANDROID_SIGNING_KEY_ALIAS_PASSWORD=...` and `!python build_colab.py --steps android --android-release` |
| Cell timed out | the nohup build is still running — re-run the watch cell |

The plain-text PAT file (`.github_pat.txt`) is gitignored — it never leaves your workspace
except as git auth headers to GitHub.
